In [1]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

df = pd.read_excel("clinic_feat_all.xlsx")

df["性别_bin"] = df["性别"].map({"男": 1, "女": 0}).astype("float32")

age_scaler = StandardScaler()
df["年龄_norm"] = age_scaler.fit_transform(df[["年龄"]])

clinic_cols = (
    ["性别_bin", "年龄_norm"] +
    [f"feat_{i}" for i in range(211)]
)

missing = [c for c in clinic_cols if c not in df.columns]
if len(missing) > 0:
    raise ValueError(f"Missing columns: {missing}")

clinic_dict = {
    str(row["id"]): row[clinic_cols].values.astype("float32")
    for _, row in df.iterrows()
}

torch.save(clinic_dict, "clinic_features.pt")

example_pid = str(df.iloc[0]["id"])
print("clinic_features.pt saved")
print(clinic_dict[example_pid].shape)


clinic_features.pt saved
(213,)


In [2]:
import pandas as pd
import torch

df = pd.read_excel("clinic_feature_selected.xlsx")

clinic_cols = [f"feat_{i}" for i in range(100)]

missing = [c for c in clinic_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

clinic_dict = {
    str(row["id"]): row[clinic_cols].values.astype("float32")
    for _, row in df.iterrows()
}

torch.save(clinic_dict, "clinic_features_selected.pt")

example_pid = str(df.iloc[0]["id"])
print("clinic_features_selected.pt saved")
print(clinic_dict[example_pid].shape)


clinic_features_selected.pt saved
(100,)


In [4]:
import pandas as pd
import torch

# -------------------------
# Load labels table
# -------------------------
labels_df = pd.read_excel("labels.xlsx")
label_ids = set(labels_df["id"].astype(str))

print("Labels table:")
print("  total rows:", len(labels_df))
print("  unique patient IDs:", len(label_ids))

# -------------------------
# Load clinic features
# -------------------------
clinic_dict = torch.load("clinic_features_selected.pt")
clinic_ids = set(clinic_dict.keys())

print("\nClinic table:")
print("  unique patient IDs:", len(clinic_ids))

# -------------------------
# Compare ID sets
# -------------------------
only_in_labels = label_ids - clinic_ids
only_in_clinic = clinic_ids - label_ids

print("\nID comparison:")
print("  IDs in labels but missing clinic:", len(only_in_labels))
print("  IDs in clinic but not in labels:", len(only_in_clinic))

# -------------------------
# Inspect mismatches
# -------------------------
if len(only_in_labels) > 0:
    print("\nSample IDs missing clinic features:")
    print(list(only_in_labels)[:10])

if len(only_in_clinic) > 0:
    print("\nSample extra clinic IDs (will be ignored):")
    print(list(only_in_clinic)[:10])

# -------------------------
# Optional: filter clinic_dict to labels only
# -------------------------
clinic_dict_filtered = {
    pid: clinic_dict[pid]
    for pid in label_ids
    if pid in clinic_dict
}

print("\nAfter filtering:")
print("  clinic_dict_filtered size:", len(clinic_dict_filtered))


Labels table:
  total rows: 211
  unique patient IDs: 211

Clinic table:
  unique patient IDs: 209

ID comparison:
  IDs in labels but missing clinic: 2
  IDs in clinic but not in labels: 0

Sample IDs missing clinic features:
['131', '308']

After filtering:
  clinic_dict_filtered size: 209


C:\Users\hyz20\AppData\Local\Temp\ipykernel_13628\3976318756.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clinic_dict = torch.load("clinic_features_selected.pt")


In [3]:
print("IDs in labels but missing clinic:", len(only_in_labels))
print("IDs in clinic but not in labels:", len(only_in_clinic))


IDs in labels but missing clinic: 2
IDs in clinic but not in labels: 0


In [5]:
import pandas as pd

# -------------------------
# Load labels table (uses 'id')
# -------------------------
labels_df = pd.read_excel("labels.xlsx")
label_ids = set(labels_df["id"].astype(str))

print("Labels table:")
print("  total rows:", len(labels_df))
print("  unique patient IDs:", len(label_ids))

# -------------------------
# Load zscore features table (uses '编号')
# -------------------------
zscore_df = pd.read_excel("zscore_feats_selected.xlsx")
zscore_ids = set(zscore_df["编号"].astype(str))

print("\nZscore features table:")
print("  total rows:", len(zscore_df))
print("  unique patient IDs:", len(zscore_ids))

# -------------------------
# Compare ID sets
# -------------------------
only_in_labels = label_ids - zscore_ids
only_in_zscore = zscore_ids - label_ids

print("\nID comparison:")
print("  IDs in labels but missing zscore:", len(only_in_labels))
print("  IDs in zscore but not in labels:", len(only_in_zscore))

# -------------------------
# Inspect mismatches
# -------------------------
if len(only_in_labels) > 0:
    print("\nSample IDs missing in zscore table:")
    print(sorted(list(only_in_labels))[:10])

if len(only_in_zscore) > 0:
    print("\nSample extra IDs in zscore table:")
    print(sorted(list(only_in_zscore))[:10])


Labels table:
  total rows: 211
  unique patient IDs: 211

Zscore features table:
  total rows: 209
  unique patient IDs: 209

ID comparison:
  IDs in labels but missing zscore: 2
  IDs in zscore but not in labels: 0

Sample IDs missing in zscore table:
['131', '308']


In [ ]:
%run train_stage1.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 100 \
  --batch_size 4 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0.5 \
  --run_name run3


In [2]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 30 \
  --batch_size 4 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0.2 \
  --run_name run3


===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 30
batch_size: 4
lr: 0.0005
weight_decay: 0.0001
auc_weight: 0.2
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43


E:\Disease-risk\train_stage2.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clinic_dict = torch.load(args.clinic_pt)
E:\Disease-risk\train_stage2.py:136: FutureWarning

Epoch 001: loss=0.6385 AUROC=0.6913 AUPRC=0.7818 Sens@95Spec=0.3043 Brier=0.2340 ✓ NEW BEST
Epoch 002: loss=0.3799 AUROC=0.6087 AUPRC=0.6573 Sens@95Spec=0.2609 Brier=0.2846
Epoch 003: loss=0.2607 AUROC=0.6739 AUPRC=0.6745 Sens@95Spec=0.3043 Brier=0.2572
Epoch 004: loss=0.2414 AUROC=0.6152 AUPRC=0.6014 Sens@95Spec=0.0870 Brier=0.3241
Epoch 005: loss=0.2173 AUROC=0.6152 AUPRC=0.6535 Sens@95Spec=0.2174 Brier=0.3186
Epoch 006: loss=0.1676 AUROC=0.5804 AUPRC=0.6388 Sens@95Spec=0.2174 Brier=0.3480
Epoch 007: loss=0.1561 AUROC=0.6217 AUPRC=0.6646 Sens@95Spec=0.2609 Brier=0.3322
Epoch 008: loss=0.1229 AUROC=0.6065 AUPRC=0.6306 Sens@95Spec=0.1739 Brier=0.3315
Epoch 009: loss=0.0976 AUROC=0.5957 AUPRC=0.6712 Sens@95Spec=0.2174 Brier=0.3558
Epoch 010: loss=0.1055 AUROC=0.5804 AUPRC=0.6364 Sens@95Spec=0.1739 Brier=0.3545
Epoch 011: loss=0.0960 AUROC=0.6022 AUPRC=0.6691 Sens@95Spec=0.2174 Brier=0.3721
Epoch 012: loss=0.0815 AUROC=0.5370 AUPRC=0.5916 Sens@95Spec=0.1739 Brier=0.4240
Epoch 013: loss=0

E:\Disease-risk\train_stage2.py:136: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(fold_dir, "mil_pool.pt"), map_location=device)


Epoch 001: loss=0.6646 AUROC=0.5273 AUPRC=0.5836 Sens@95Spec=0.0909 Brier=0.3175 ✓ NEW BEST
Epoch 002: loss=0.3969 AUROC=0.5386 AUPRC=0.5625 Sens@95Spec=0.0909 Brier=0.3247 ✓ NEW BEST
Epoch 003: loss=0.2747 AUROC=0.5500 AUPRC=0.5829 Sens@95Spec=0.1364 Brier=0.3193 ✓ NEW BEST
Epoch 004: loss=0.2149 AUROC=0.5682 AUPRC=0.5970 Sens@95Spec=0.1364 Brier=0.3141 ✓ NEW BEST
Epoch 005: loss=0.1774 AUROC=0.5750 AUPRC=0.5915 Sens@95Spec=0.1818 Brier=0.3333 ✓ NEW BEST
Epoch 006: loss=0.1516 AUROC=0.5750 AUPRC=0.5984 Sens@95Spec=0.1818 Brier=0.3312
Epoch 007: loss=0.1444 AUROC=0.5977 AUPRC=0.6076 Sens@95Spec=0.1818 Brier=0.3236 ✓ NEW BEST
Epoch 008: loss=0.1303 AUROC=0.5909 AUPRC=0.5840 Sens@95Spec=0.0909 Brier=0.3531
Epoch 009: loss=0.1511 AUROC=0.5773 AUPRC=0.5700 Sens@95Spec=0.0000 Brier=0.3872
Epoch 010: loss=0.1235 AUROC=0.5886 AUPRC=0.6023 Sens@95Spec=0.0909 Brier=0.3740
Epoch 011: loss=0.0965 AUROC=0.5523 AUPRC=0.5863 Sens@95Spec=0.1364 Brier=0.3870
Epoch 012: loss=0.0788 AUROC=0.5386 AUPRC=0

...

Epoch 001: loss=0.6801 AUROC=0.7500 AUPRC=0.7837 Sens@95Spec=0.4091 Brier=0.2074 ✓ NEW BEST
Epoch 002: loss=0.3926 AUROC=0.6727 AUPRC=0.6834 Sens@95Spec=0.1364 Brier=0.2354
Epoch 003: loss=0.3935 AUROC=0.7091 AUPRC=0.7068 Sens@95Spec=0.1818 Brier=0.2260
Epoch 004: loss=0.2540 AUROC=0.6909 AUPRC=0.7071 Sens@95Spec=0.1818 Brier=0.2368
Epoch 005: loss=0.2048 AUROC=0.6841 AUPRC=0.7003 Sens@95Spec=0.1364 Brier=0.2524
Epoch 006: loss=0.1840 AUROC=0.6841 AUPRC=0.6892 Sens@95Spec=0.1818 Brier=0.2411
Epoch 007: loss=0.1650 AUROC=0.6750 AUPRC=0.7284 Sens@95Spec=0.2273 Brier=0.2757
Epoch 008: loss=0.1266 AUROC=0.6500 AUPRC=0.6807 Sens@95Spec=0.1364 Brier=0.2965
Epoch 009: loss=0.1215 AUROC=0.6386 AUPRC=0.6719 Sens@95Spec=0.1364 Brier=0.2974
Epoch 010: loss=0.0980 AUROC=0.6659 AUPRC=0.6977 Sens@95Spec=0.2727 Brier=0.2874
Epoch 011: loss=0.1024 AUROC=0.5909 AUPRC=0.6325 Sens@95Spec=0.1364 Brier=0.3144
Epoch 012: loss=0.0919 AUROC=0.6273 AUPRC=0.6742 Sens@95Spec=0.1818 Brier=0.3076
Epoch 013: loss=0

E:\Disease-risk\train_stage2.py:136: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(fold_dir, "mil_pool.pt"), map_location=device)


Epoch 001: loss=0.6813 AUROC=0.6794 AUPRC=0.7036 Sens@95Spec=0.0909 Brier=0.2436 ✓ NEW BEST
Epoch 002: loss=0.4629 AUROC=0.6531 AUPRC=0.6577 Sens@95Spec=0.0455 Brier=0.2470
Epoch 003: loss=0.3033 AUROC=0.8014 AUPRC=0.7897 Sens@95Spec=0.0455 Brier=0.1916 ✓ NEW BEST
Epoch 004: loss=0.2296 AUROC=0.7919 AUPRC=0.7430 Sens@95Spec=0.0000 Brier=0.1836
Epoch 005: loss=0.2181 AUROC=0.7871 AUPRC=0.7624 Sens@95Spec=0.0000 Brier=0.1963
Epoch 006: loss=0.1574 AUROC=0.7871 AUPRC=0.7525 Sens@95Spec=0.0000 Brier=0.1870
Epoch 007: loss=0.1493 AUROC=0.7536 AUPRC=0.7212 Sens@95Spec=0.0000 Brier=0.2088
Epoch 008: loss=0.1267 AUROC=0.7536 AUPRC=0.7163 Sens@95Spec=0.0000 Brier=0.2062
Epoch 009: loss=0.1212 AUROC=0.7488 AUPRC=0.7008 Sens@95Spec=0.0000 Brier=0.1942
Epoch 010: loss=0.0924 AUROC=0.7835 AUPRC=0.7266 Sens@95Spec=0.0000 Brier=0.1801
Epoch 011: loss=0.0935 AUROC=0.8026 AUPRC=0.7542 Sens@95Spec=0.0000 Brier=0.1788 ✓ NEW BEST
Epoch 012: loss=0.0699 AUROC=0.7500 AUPRC=0.7234 Sens@95Spec=0.0000 Brier=0.

E:\Disease-risk\train_stage2.py:136: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(fold_dir, "mil_pool.pt"), map_location=device)


Epoch 001: loss=0.6765 AUROC=0.6476 AUPRC=0.7108 Sens@95Spec=0.2857 Brier=0.2341 ✓ NEW BEST
Epoch 002: loss=0.3612 AUROC=0.7119 AUPRC=0.7773 Sens@95Spec=0.3810 Brier=0.2235 ✓ NEW BEST
Epoch 003: loss=0.3079 AUROC=0.5976 AUPRC=0.6454 Sens@95Spec=0.3333 Brier=0.3017
Epoch 004: loss=0.2520 AUROC=0.6786 AUPRC=0.7267 Sens@95Spec=0.4286 Brier=0.2695
Epoch 005: loss=0.2243 AUROC=0.6690 AUPRC=0.7373 Sens@95Spec=0.4762 Brier=0.2730
Epoch 006: loss=0.1762 AUROC=0.6381 AUPRC=0.6790 Sens@95Spec=0.3810 Brier=0.2977
Epoch 007: loss=0.1525 AUROC=0.6690 AUPRC=0.7559 Sens@95Spec=0.4762 Brier=0.2863
Epoch 008: loss=0.1230 AUROC=0.7048 AUPRC=0.7658 Sens@95Spec=0.5238 Brier=0.2739
Epoch 009: loss=0.1116 AUROC=0.6833 AUPRC=0.7618 Sens@95Spec=0.5238 Brier=0.2933
Epoch 010: loss=0.0962 AUROC=0.6595 AUPRC=0.7400 Sens@95Spec=0.4762 Brier=0.3068
Epoch 011: loss=0.0950 AUROC=0.6405 AUPRC=0.6765 Sens@95Spec=0.3333 Brier=0.3275
Epoch 012: loss=0.1156 AUROC=0.7048 AUPRC=0.7823 Sens@95Spec=0.4286 Brier=0.3056
Epoch 

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 30 \
  --batch_size 4 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 30
batch_size: 4
lr: 0.0005
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43


E:\Disease-risk\train_stage2.py:83: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clinic_dict = torch.load(args.clinic_pt)


Epoch 001: loss=0.7716 AUROC=0.6913 AUPRC=0.7802 Sens@95Spec=0.3478 Brier=0.2327 ✓ NEW BEST
Epoch 002: loss=0.4448 AUROC=0.6109 AUPRC=0.6590 Sens@95Spec=0.2609 Brier=0.2837
Epoch 003: loss=0.2947 AUROC=0.6652 AUPRC=0.6695 Sens@95Spec=0.3043 Brier=0.2587
Epoch 004: loss=0.2643 AUROC=0.6087 AUPRC=0.6015 Sens@95Spec=0.0870 Brier=0.3206
Epoch 005: loss=0.2287 AUROC=0.6196 AUPRC=0.6529 Sens@95Spec=0.2174 Brier=0.3184
Epoch 006: loss=0.1698 AUROC=0.5804 AUPRC=0.6394 Sens@95Spec=0.2174 Brier=0.3492
Epoch 007: loss=0.1550 AUROC=0.6152 AUPRC=0.6482 Sens@95Spec=0.2609 Brier=0.3316
Epoch 008: loss=0.1107 AUROC=0.6065 AUPRC=0.6362 Sens@95Spec=0.2174 Brier=0.3313
Epoch 009: loss=0.0802 AUROC=0.5978 AUPRC=0.6720 Sens@95Spec=0.2174 Brier=0.3561
Epoch 010: loss=0.0885 AUROC=0.5804 AUPRC=0.6362 Sens@95Spec=0.1739 Brier=0.3545
Epoch 011: loss=0.0759 AUROC=0.6022 AUPRC=0.6718 Sens@95Spec=0.2174 Brier=0.3722
Epoch 012: loss=0.0561 AUROC=0.5478 AUPRC=0.6097 Sens@95Spec=0.1739 Brier=0.4211
Epoch 013: loss=0

In [2]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 30 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 30
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6583 AUROC=0.6065 AUPRC=0.6094 Sens@95Spec=0.0000 Brier=0.2472 ✓ NEW BEST
Epoch 002: loss=0.5271 AUROC=0.5935 AUPRC=0.6181 Sens@95Spec=0.0870 Brier=0.2613
Epoch 003: loss=0.4784 AUROC=0.6391 AUPRC=0.6278 Sens@95Spec=0.0435 Brier=0.2549 ✓ NEW BEST
Epoch 004: loss=0.4282 AUROC=0.6043 AUPRC=0.6474 Sens@95Spec=0.2609 Brier=0.2573
Epoch 005: loss=0.3853 AUROC=0.6457 AUPRC=0.6550 Sens@95Spec=0.1739 Brier=0.2457 ✓ NEW BEST
Epoch 006: loss=0.3595 AUROC=0.6087 AUPRC=0.6514 Sens@95Spec=0.3043 Brier=0.2667
Epoch 007: loss=0.3330 AUROC=0.6348 AUPRC=0.6638 Sens@95Spec=0.3043 Brier=0.2528
Epoch 008: 

In [3]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 30 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy fixed_random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: fixed_random
use_combined_loss: True
epochs: 30
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6709 AUROC=0.6196 AUPRC=0.6437 Sens@95Spec=0.0000 Brier=0.2483 ✓ NEW BEST
Epoch 002: loss=0.5422 AUROC=0.6022 AUPRC=0.6234 Sens@95Spec=0.0000 Brier=0.2602
Epoch 003: loss=0.4526 AUROC=0.6196 AUPRC=0.6688 Sens@95Spec=0.0870 Brier=0.2526
Epoch 004: loss=0.4096 AUROC=0.6065 AUPRC=0.6469 Sens@95Spec=0.2174 Brier=0.2588
Epoch 005: loss=0.3608 AUROC=0.6304 AUPRC=0.7030 Sens@95Spec=0.3478 Brier=0.2584 ✓ NEW BEST
Epoch 006: loss=0.3292 AUROC=0.6217 AUPRC=0.6637 Sens@95Spec=0.3478 Brier=0.2595
Epoch 007: loss=0.2998 AUROC=0.5935 AUPRC=0.6317 Sens@95Spec=0.2609 Brier=0.2682
Epoch 008: loss=

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 30 \
  --batch_size 4 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 30
batch_size: 4
lr: 0.0005
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.7259 AUROC=0.5304 AUPRC=0.5772 Sens@95Spec=0.0435 Brier=0.2914 ✓ NEW BEST
Epoch 002: loss=0.4701 AUROC=0.6109 AUPRC=0.6204 Sens@95Spec=0.0870 Brier=0.2645 ✓ NEW BEST
Epoch 003: loss=0.3100 AUROC=0.6022 AUPRC=0.6581 Sens@95Spec=0.2174 Brier=0.2807
Epoch 004: loss=0.2143 AUROC=0.6043 AUPRC=0.5988 Sens@95Spec=0.0870 Brier=0.2660
Epoch 005: loss=0.1480 AUROC=0.6043 AUPRC=0.6694 Sens@95Spec=0.1739 Brier=0.3208
Epoch 006: loss=0.0955 AUROC=0.6174 AUPRC=0.6299 Sens@95Spec=0.1304 Brier=0.2983 ✓ NEW BEST
Epoch 007: loss=0.0636 AUROC=0.6304 AUPRC=0.6209 Sens@95Spec=0.1304 Brier=0.2938 ✓ NEW BEST


In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 30 \
  --batch_size 4 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 30
batch_size: 4
lr: 0.0005
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.8078 AUROC=0.6152 AUPRC=0.6976 Sens@95Spec=0.2174 Brier=0.2620 ✓ NEW BEST
Epoch 002: loss=0.4090 AUROC=0.6174 AUPRC=0.6293 Sens@95Spec=0.0870 Brier=0.2750 ✓ NEW BEST
Epoch 003: loss=0.2879 AUROC=0.6870 AUPRC=0.6854 Sens@95Spec=0.1304 Brier=0.2624 ✓ NEW BEST
Epoch 004: loss=0.2599 AUROC=0.6630 AUPRC=0.6605 Sens@95Spec=0.1304 Brier=0.2537
Epoch 005: loss=0.1930 AUROC=0.6478 AUPRC=0.6803 Sens@95Spec=0.2174 Brier=0.2849
Epoch 006: loss=0.1472 AUROC=0.6043 AUPRC=0.6738 Sens@95Spec=0.2174 Brier=0.3259
Epoch 007: loss=0.1639 AUROC=0.6413 AUPRC=0.7269 Sens@95Spec=0.3043 Brier=0.3265
Epoch 008: 

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 30 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 30
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6828 AUROC=0.5348 AUPRC=0.6117 Sens@95Spec=0.2174 Brier=0.2546 ✓ NEW BEST
Epoch 002: loss=0.5492 AUROC=0.5978 AUPRC=0.6270 Sens@95Spec=0.0435 Brier=0.2533 ✓ NEW BEST
Epoch 003: loss=0.4810 AUROC=0.6435 AUPRC=0.6637 Sens@95Spec=0.0870 Brier=0.2448 ✓ NEW BEST
Epoch 004: loss=0.4210 AUROC=0.6891 AUPRC=0.7318 Sens@95Spec=0.1739 Brier=0.2288 ✓ NEW BEST
Epoch 005: loss=0.3621 AUROC=0.6891 AUPRC=0.7105 Sens@95Spec=0.2609 Brier=0.2183
Epoch 006: loss=0.3488 AUROC=0.5870 AUPRC=0.6436 Sens@95Spec=0.2174 Brier=0.2628
Epoch 007: loss=0.3467 AUROC=0.6217 AUPRC=0.6964 Sens@95Spec=0.3043 Brier=0.2510


In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 70 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 70
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6828 AUROC=0.5348 AUPRC=0.6117 Sens@95Spec=0.2174 Brier=0.2546 ✓ NEW BEST
Epoch 002: loss=0.5492 AUROC=0.5978 AUPRC=0.6270 Sens@95Spec=0.0435 Brier=0.2533 ✓ NEW BEST
Epoch 003: loss=0.4810 AUROC=0.6435 AUPRC=0.6637 Sens@95Spec=0.0870 Brier=0.2448 ✓ NEW BEST
Epoch 004: loss=0.4210 AUROC=0.6891 AUPRC=0.7318 Sens@95Spec=0.1739 Brier=0.2288 ✓ NEW BEST
Epoch 005: loss=0.3621 AUROC=0.6891 AUPRC=0.7105 Sens@95Spec=0.2609 Brier=0.2183
Epoch 006: loss=0.3488 AUROC=0.5870 AUPRC=0.6436 Sens@95Spec=0.2174 Brier=0.2628
Epoch 007: loss=0.3467 AUROC=0.6217 AUPRC=0.6964 Sens@95Spec=0.3043 Brier=0.2510


In [ ]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 70 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 70 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 70
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6511 AUROC=0.5761 AUPRC=0.6528 Sens@95Spec=0.1739 Brier=0.2410 ✓ NEW BEST
Epoch 002: loss=0.5596 AUROC=0.5043 AUPRC=0.6434 Sens@95Spec=0.2174 Brier=0.2618
Epoch 003: loss=0.5313 AUROC=0.5761 AUPRC=0.6014 Sens@95Spec=0.0435 Brier=0.2712
Epoch 004: loss=0.4724 AUROC=0.5087 AUPRC=0.6320 Sens@95Spec=0.1739 Brier=0.2827
Epoch 005: loss=0.4190 AUROC=0.5152 AUPRC=0.5399 Sens@95Spec=0.0435 Brier=0.2673
Epoch 006: loss=0.3492 AUROC=0.4935 AUPRC=0.5503 Sens@95Spec=0.0870 Brier=0.2902
Epoch 007: loss=0.3173 AUROC=0.5891 AUPRC=0.6545 Sens@95Spec=0.1739 Brier=0.2711 ✓ NEW BEST
Epoch 008: loss=0.2679

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 70 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 70
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6443 AUROC=0.4891 AUPRC=0.5551 Sens@95Spec=0.0870 Brier=0.2602 ✓ NEW BEST
Epoch 002: loss=0.5499 AUROC=0.6652 AUPRC=0.6682 Sens@95Spec=0.0435 Brier=0.2289 ✓ NEW BEST
Epoch 003: loss=0.4493 AUROC=0.5804 AUPRC=0.6869 Sens@95Spec=0.3043 Brier=0.2485
Epoch 004: loss=0.4251 AUROC=0.5478 AUPRC=0.5973 Sens@95Spec=0.0870 Brier=0.2636
Epoch 005: loss=0.3807 AUROC=0.5652 AUPRC=0.6207 Sens@95Spec=0.1739 Brier=0.2763
Epoch 006: loss=0.3062 AUROC=0.6565 AUPRC=0.7026 Sens@95Spec=0.2174 Brier=0.2550
Epoch 007: loss=0.2927 AUROC=0.5957 AUPRC=0.6720 Sens@95Spec=0.1739 Brier=0.2634
Epoch 008: loss=0.2330

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 70 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0.2 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 70
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.2
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.5339 AUROC=0.4935 AUPRC=0.5576 Sens@95Spec=0.0870 Brier=0.2586 ✓ NEW BEST
Epoch 002: loss=0.4608 AUROC=0.6652 AUPRC=0.6734 Sens@95Spec=0.0870 Brier=0.2290 ✓ NEW BEST
Epoch 003: loss=0.3805 AUROC=0.5783 AUPRC=0.6874 Sens@95Spec=0.3043 Brier=0.2463
Epoch 004: loss=0.3674 AUROC=0.5391 AUPRC=0.5918 Sens@95Spec=0.0870 Brier=0.2638
Epoch 005: loss=0.3296 AUROC=0.5783 AUPRC=0.6315 Sens@95Spec=0.1739 Brier=0.2744
Epoch 006: loss=0.2732 AUROC=0.6609 AUPRC=0.7072 Sens@95Spec=0.2174 Brier=0.2504
Epoch 007: loss=0.2618 AUROC=0.6043 AUPRC=0.6784 Sens@95Spec=0.1739 Brier=0.2606
Epoch 008: loss=0.2183

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 100 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 100
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6443 AUROC=0.4891 AUPRC=0.5551 Sens@95Spec=0.0870 Brier=0.2602 ✓ NEW BEST
Epoch 002: loss=0.5499 AUROC=0.6652 AUPRC=0.6682 Sens@95Spec=0.0435 Brier=0.2289 ✓ NEW BEST
Epoch 003: loss=0.4493 AUROC=0.5804 AUPRC=0.6869 Sens@95Spec=0.3043 Brier=0.2485
Epoch 004: loss=0.4251 AUROC=0.5478 AUPRC=0.5973 Sens@95Spec=0.0870 Brier=0.2636
Epoch 005: loss=0.3807 AUROC=0.5652 AUPRC=0.6207 Sens@95Spec=0.1739 Brier=0.2763
Epoch 006: loss=0.3062 AUROC=0.6565 AUPRC=0.7026 Sens@95Spec=0.2174 Brier=0.2550
Epoch 007: loss=0.2927 AUROC=0.5957 AUPRC=0.6720 Sens@95Spec=0.1739 Brier=0.2634
Epoch 008: loss=0.233

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 100 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 100
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6443 AUROC=0.4891 AUPRC=0.5551 Sens@95Spec=0.0870 Brier=0.2602 ✓ NEW BEST
Epoch 002: loss=0.5499 AUROC=0.6652 AUPRC=0.6682 Sens@95Spec=0.0435 Brier=0.2289 ✓ NEW BEST
Epoch 003: loss=0.4493 AUROC=0.5804 AUPRC=0.6869 Sens@95Spec=0.3043 Brier=0.2485
Epoch 004: loss=0.4251 AUROC=0.5478 AUPRC=0.5973 Sens@95Spec=0.0870 Brier=0.2636
Epoch 005: loss=0.3807 AUROC=0.5652 AUPRC=0.6207 Sens@95Spec=0.1739 Brier=0.2763
Epoch 006: loss=0.3062 AUROC=0.6565 AUPRC=0.7026 Sens@95Spec=0.2174 Brier=0.2550
Epoch 007: loss=0.2927 AUROC=0.5957 AUPRC=0.6720 Sens@95Spec=0.1739 Brier=0.2634
Epoch 008: loss=0.233

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 100 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0.3 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 100
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.3
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.4777 AUROC=0.4826 AUPRC=0.5519 Sens@95Spec=0.0870 Brier=0.2577 ✓ NEW BEST
Epoch 002: loss=0.4151 AUROC=0.6696 AUPRC=0.6794 Sens@95Spec=0.0870 Brier=0.2291 ✓ NEW BEST
Epoch 003: loss=0.3481 AUROC=0.5826 AUPRC=0.6913 Sens@95Spec=0.3043 Brier=0.2446
Epoch 004: loss=0.3364 AUROC=0.5391 AUPRC=0.5911 Sens@95Spec=0.0870 Brier=0.2620
Epoch 005: loss=0.3026 AUROC=0.5761 AUPRC=0.6288 Sens@95Spec=0.1739 Brier=0.2756
Epoch 006: loss=0.2573 AUROC=0.6630 AUPRC=0.7108 Sens@95Spec=0.2174 Brier=0.2477
Epoch 007: loss=0.2473 AUROC=0.6065 AUPRC=0.6826 Sens@95Spec=0.1739 Brier=0.2603
Epoch 008: loss=0.213

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 100 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 100
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6443 AUROC=0.4891 AUPRC=0.5551 Sens@95Spec=0.0870 Brier=0.2602 ✓ NEW BEST
Epoch 002: loss=0.5499 AUROC=0.6652 AUPRC=0.6682 Sens@95Spec=0.0435 Brier=0.2289 ✓ NEW BEST
Epoch 003: loss=0.4493 AUROC=0.5804 AUPRC=0.6869 Sens@95Spec=0.3043 Brier=0.2485
Epoch 004: loss=0.4251 AUROC=0.5478 AUPRC=0.5973 Sens@95Spec=0.0870 Brier=0.2636
Epoch 005: loss=0.3807 AUROC=0.5652 AUPRC=0.6207 Sens@95Spec=0.1739 Brier=0.2763
Epoch 006: loss=0.3062 AUROC=0.6565 AUPRC=0.7026 Sens@95Spec=0.2174 Brier=0.2550
Epoch 007: loss=0.2927 AUROC=0.5957 AUPRC=0.6720 Sens@95Spec=0.1739 Brier=0.2634
Epoch 008: loss=0.233

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features_selected.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 100 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features_selected.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 100
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6292 AUROC=0.4174 AUPRC=0.5447 Sens@95Spec=0.0870 Brier=0.2905 ✓ NEW BEST
Epoch 002: loss=0.5660 AUROC=0.3761 AUPRC=0.4715 Sens@95Spec=0.0000 Brier=0.2886
Epoch 003: loss=0.4644 AUROC=0.4739 AUPRC=0.5612 Sens@95Spec=0.0870 Brier=0.2955 ✓ NEW BEST
Epoch 004: loss=0.4862 AUROC=0.5717 AUPRC=0.6645 Sens@95Spec=0.2174 Brier=0.2742 ✓ NEW BEST
Epoch 005: loss=0.4778 AUROC=0.6761 AUPRC=0.7379 Sens@95Spec=0.3478 Brier=0.2307 ✓ NEW BEST
Epoch 006: loss=0.3929 AUROC=0.6500 AUPRC=0.6548 Sens@95Spec=0.0870 Brier=0.2651
Epoch 007: loss=0.3829 AUROC=0.5196 AUPRC=0.6100 Sens@95Spec=0.1739 Bri

In [1]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 70 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3

===== STAGE-2 CONFIG =====
feat_dir: ./encoder_features
labels_csv: ./labels.xlsx
clinic_pt: ./clinic_features.pt
run_name: run3
label_cols: ['内分泌代谢疾病']
architecture: attention
max_feats: 16
instance_strategy: random
use_combined_loss: True
epochs: 70
batch_size: 4
lr: 0.0001
weight_decay: 0.0001
auc_weight: 0.0
folds: 5
seed: 42


===== Stage-2 Fold 1/5 =====
Fold 1: train=166, val=43
Epoch 001: loss=0.6443 AUROC=0.4891 AUPRC=0.5551 Sens@95Spec=0.0870 Brier=0.2602 ✓ NEW BEST
Epoch 002: loss=0.5499 AUROC=0.6652 AUPRC=0.6682 Sens@95Spec=0.0435 Brier=0.2289 ✓ NEW BEST
Epoch 003: loss=0.4493 AUROC=0.5804 AUPRC=0.6869 Sens@95Spec=0.3043 Brier=0.2485
Epoch 004: loss=0.4251 AUROC=0.5478 AUPRC=0.5973 Sens@95Spec=0.0870 Brier=0.2636
Epoch 005: loss=0.3807 AUROC=0.5652 AUPRC=0.6207 Sens@95Spec=0.1739 Brier=0.2763
Epoch 006: loss=0.3062 AUROC=0.6565 AUPRC=0.7026 Sens@95Spec=0.2174 Brier=0.2550
Epoch 007: loss=0.2927 AUROC=0.5957 AUPRC=0.6720 Sens@95Spec=0.1739 Brier=0.2634
Epoch 008: loss=0.2330